# Task 1

In [ ]:
'''
1. What does the **Embedding** layer do to a word index?
It converts discrete integer word indices (like token IDs) into dense, continuous vectors of a fixed size. 
This allows the network to learn semantic relationships and similarities between words.

2. What is the role of the **Encoder LSTM**?
It reads and processes the input sequence (source language) step-by-step, capturing the contextual information and summarizing 
the entire input sentence into a final hidden/thought state.

3. What does the **Decoder LSTM** do?
It takes the summary context vector from the encoder and sequentially generates the output sequence (target language) word-by-word.

4. Why do we use `softmax` in the final layer?
It converts the raw model output scores (logits) into a probability distribution across the entire target vocabulary, allowing us 
to select the most likely translated word (the one with the highest probability).
'''


# Task 2

In [2]:
!pip install tensorflow

   ---------------------------------------- 0.0/351.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/351.2 MB ? eta -:--:--
   ---------------------------------------- 1.0/351.2 MB 4.1 MB/s eta 0:01:26
   ---------------------------------------- 2.9/351.2 MB 6.9 MB/s eta 0:00:51
   ---------------------------------------- 3.7/351.2 MB 6.1 MB/s eta 0:00:57
   ---------------------------------------- 3.7/351.2 MB 6.1 MB/s eta 0:00:57
    --------------------------------------- 5.0/351.2 MB 4.8 MB/s eta 0:01:12
    --------------------------------------- 6.0/351.2 MB 5.2 MB/s eta 0:01:08
    --------------------------------------- 6.3/351.2 MB 4.1 MB/s eta 0:01:24
    --------------------------------------- 8.7/351.2 MB 5.0 MB/s eta 0:01:09
   - -------------------------------------- 8.9/351.2 MB 5.0 MB/s eta 0:01:09
   - -------------------------------------- 10.7/351.2 MB 4.9 MB/s eta 0:01:10
   - -------------------------------------- 12.1/351.2 MB 5.0 MB/s eta 0:01:0

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.45.1 requires protobuf<7,>=3.20, but you have protobuf 7.36.2 which is incompatible.


In [5]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
import numpy as np

# Original lab bilingual dataset
english_sentences = ["Hello.", "How are you?", "This is a machine translation example."]
french_sentences = ["Bonjour.", "Comment ça va?", "Ceci est un exemple de traduction automatique."]

# 1. Tokenization and Padding
tokenizer_eng = Tokenizer()
tokenizer_eng.fit_on_texts(english_sentences)
eng_sequences = tokenizer_eng.texts_to_sequences(english_sentences)
eng_padded = pad_sequences(eng_sequences)

tokenizer_frn = Tokenizer()
tokenizer_frn.fit_on_texts(french_sentences)
frn_sequences = tokenizer_frn.texts_to_sequences(french_sentences)
frn_padded = pad_sequences(frn_sequences, maxlen=eng_padded.shape[1])

# 2. Train-test split
eng_train, eng_test, frn_train, frn_test = train_test_split(eng_padded, frn_padded, test_size=0.2, random_state=42)

# 3. Define the model
model = tf.keras.Sequential([
    tf.keras.layers.Embedding(input_dim=len(tokenizer_eng.word_index) + 1, output_dim=64),
    tf.keras.layers.LSTM(64),
    tf.keras.layers.RepeatVector(frn_padded.shape[1]),
    tf.keras.layers.LSTM(64, return_sequences=True),
    tf.keras.layers.TimeDistributed(tf.keras.layers.Dense(len(tokenizer_frn.word_index) + 1, activation='softmax'))
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# 4. Train the model (using 50 epochs matching the original lab notebook)
model.fit(eng_train, np.expand_dims(frn_train, -1), epochs=50, batch_size=1, verbose=0)

# 5. Inference function
def translate_sentence(sentence, model, eng_tokenizer, frn_tokenizer):
    sequence = eng_tokenizer.texts_to_sequences([sentence])
    padded_sequence = pad_sequences(sequence, maxlen=eng_padded.shape[1])
    predicted_sequence = model.predict(padded_sequence, verbose=0)
    translated_sequence = np.argmax(predicted_sequence, axis=-1)
    translated_sentence = frn_tokenizer.sequences_to_texts(translated_sequence)
    return translated_sentence[0]

# Test using the exact lab sentence
input_sentence = "How are you?"
translated_sentence = translate_sentence(input_sentence, model, tokenizer_eng, tokenizer_frn)

print(f"E-Commerce Input (English): {input_sentence}")
print(f"Translated Output (French): {translated_sentence}")

E-Commerce Input (English): How are you?
Translated Output (French): comment ça va
